<a href="https://colab.research.google.com/github/madhumitha-murthy/cs-llm-finetuning/blob/main/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Customer Service LLM — Full Training Pipeline
**Qwen2.5-0.5B | SFT → DPO → INT4 Quantisation**

Run each cell in order. Use Colab T4 GPU (free tier).

Stages:
1. Install dependencies
2. SFT with LoRA
3. Evaluate base vs SFT
4. DPO alignment
5. Evaluate SFT vs DPO
6. INT4 Quantisation + latency benchmark
7. Save results → update CV

In [1]:
# CELL 1 — Install dependencies (pinned compatible versions)
!pip install -q -U transformers==4.44.0 peft==0.12.0 trl==0.10.1 \
    datasets bitsandbytes accelerate==0.34.0 \
    evaluate rouge-score bert-score mlflow sentencepiece scipy
print('Done')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 65.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.1/280.1 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.8 MB/s eta 0:00:00
 

In [2]:
# CELL 2 — Check GPU
import torch
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

GPU: Tesla T4
VRAM: 15.6 GB


In [3]:
# CELL 3 — Load dataset and inspect
from datasets import load_dataset

ds = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset')
print(ds)
print('\nSample:')
print('Instruction:', ds['train'][0]['instruction'])
print('Response   :', ds['train'][0]['response'])
print('Intent     :', ds['train'][0]['intent'])
print('\nUnique intents:', len(set(ds['train']['intent'])))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Bitext_Sample_Customer_Support_Training_(…):   0%|          | 0.00/19.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 26872
    })
})

Sample:
Instruction: question about cancelling order {{Order Number}}
Response   : I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you.
Intent     : cancel_order

Unique intents: 27


In [4]:
# CELL 4 — Format dataset as chat template
def format_example(example):
    return {
        'text': (
            '<|im_start|>system\n'
            'You are a helpful e-commerce customer service assistant. '
            'Answer customer queries about orders, refunds, shipping, and disputes accurately.\n<|im_end|>\n'
            f'<|im_start|>user\n{example["instruction"]}\n<|im_end|>\n'
            f'<|im_start|>assistant\n{example["response"]}\n<|im_end|>'
        )
    }

ds = ds['train'].train_test_split(test_size=0.1, seed=42)
train_ds = ds['train'].map(format_example, remove_columns=ds['train'].column_names)
eval_ds  = ds['test'].map(format_example, remove_columns=ds['test'].column_names)

print('Train size:', len(train_ds))
print('Eval size :', len(eval_ds))
print('\nFormatted example:')
print(train_ds[0]['text'][:300])

Map:   0%|          | 0/24184 [00:00<?, ? examples/s]

Map:   0%|          | 0/2688 [00:00<?, ? examples/s]

Train size: 24184
Eval size : 2688

Formatted example:
<|im_start|>system
You are a helpful e-commerce customer service assistant. Answer customer queries about orders, refunds, shipping, and disputes accurately.
<|im_end|>
<|im_start|>user
could uhelp me close a freemium account
<|im_end|>
<|im_start|>assistant
Thank you for your message to us for assi


In [5]:
# CELL 5 — Load model + LoRA config
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [6]:
# CELL 6 — SFT Training
import mlflow
from transformers import TrainingArguments
from trl import SFTTrainer

mlflow.set_experiment('cs-llm-finetuning')

training_args = TrainingArguments(
    output_dir='./sft_output',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=50,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='mlflow',
    run_name='sft-qwen2.5-0.5b',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field='text',
    max_seq_length=512,
    args=training_args,
)

with mlflow.start_run(run_name='sft'):
    mlflow.log_params({'lora_r': 16, 'lora_alpha': 32, 'epochs': 3, 'lr': 2e-4})
    trainer.train()

trainer.save_model('./sft_output/final')
tokenizer.save_pretrained('./sft_output/final')
print('SFT training complete. Model saved.')

2026/04/09 14:52:05 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/09 14:52:05 INFO mlflow.store.db.utils: Updating database tables
2026/04/09 14:52:07 INFO mlflow.tracking.fluent: Experiment with name 'cs-llm-finetuning' does not exist. Creating a new experiment.
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_

Map:   0%|          | 0/24184 [00:00<?, ? examples/s]

Map:   0%|          | 0/2688 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss
0,0.578600,0.576171
2,0.469100,0.531997


SFT training complete. Model saved.


In [7]:
# CELL 7 — Quick evaluation: base vs SFT
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn
from peft import PeftModel
import time

raw_ds = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset', split='train[:100]')
test_prompts = [
    '<|im_start|>system\nYou are a helpful e-commerce customer service assistant.\n<|im_end|>\n'
    f'<|im_start|>user\n{ex["instruction"]}\n<|im_end|>\n<|im_start|>assistant\n'
    for ex in raw_ds
]
references = [ex['response'] for ex in raw_ds]

def evaluate(mdl, tok, prompts, refs, name):
    preds, lats = [], []
    mdl.eval()
    for p in prompts:
        inp = tok(p, return_tensors='pt').to(mdl.device)
        t = time.time()
        with torch.no_grad():
            out = mdl.generate(**inp, max_new_tokens=150, do_sample=False, pad_token_id=tok.eos_token_id)
        lats.append(time.time() - t)
        preds.append(tok.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip())

    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rouge_l = sum(scorer.score(r,p)['rougeL'].fmeasure for p,r in zip(preds, refs)) / len(preds)
    _, _, bs_f1 = bert_score_fn(preds, refs, lang='en', verbose=False)

    print(f'\n{name}: ROUGE-L={rouge_l:.4f} | BERTScore={bs_f1.mean():.4f} | Avg latency={sum(lats)/len(lats)*1000:.0f}ms')
    return preds, {'rouge_l': round(rouge_l,4), 'bertscore': round(bs_f1.mean().item(),4)}

# Base model
base_mdl = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
_, base_scores = evaluate(base_mdl, tokenizer, test_prompts, references, 'BASE')
del base_mdl; torch.cuda.empty_cache()

# SFT model
sft_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
sft_mdl = PeftModel.from_pretrained(sft_base, './sft_output/final')
_, sft_scores = evaluate(sft_mdl, tokenizer, test_prompts, references, 'SFT')
del sft_mdl, sft_base; torch.cuda.empty_cache()

print('\n--- WRITE THESE DOWN ---')
print('Base ROUGE-L:', base_scores['rouge_l'], '| BERTScore:', base_scores['bertscore'])
print('SFT  ROUGE-L:', sft_scores['rouge_l'],  '| BERTScore:', sft_scores['bertscore'])

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:589: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



BASE: ROUGE-L=0.1088 | BERTScore=0.8292 | Avg latency=1224ms


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



SFT: ROUGE-L=0.4092 | BERTScore=0.9089 | Avg latency=9865ms

--- WRITE THESE DOWN ---
Base ROUGE-L: 0.1088 | BERTScore: 0.8292
SFT  ROUGE-L: 0.4092 | BERTScore: 0.9089


In [ ]:
# CELL 8 — Generate DPO preference pairs
from datasets import Dataset

dpo_raw = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset', split='train[:2000]')

def build_prompt(instruction):
    return (
        '<|im_start|>system\nYou are a helpful e-commerce customer service assistant.\n<|im_end|>\n'
        f'<|im_start|>user\n{instruction}\n<|im_end|>\n<|im_start|>assistant\n'
    )

# Load base (rejected) and SFT (chosen)
base_mdl = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
sft_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
sft_mdl  = PeftModel.from_pretrained(sft_base, './sft_output/final')

pairs = []
for ex in dpo_raw:
    prompt = build_prompt(ex['instruction'])
    inp = tokenizer(prompt, return_tensors='pt').to(base_mdl.device)

    with torch.no_grad():
        rej_out = base_mdl.generate(**inp, max_new_tokens=150, do_sample=True, temperature=0.9, pad_token_id=tokenizer.eos_token_id)
        cho_out = sft_mdl.generate(**inp, max_new_tokens=150, do_sample=False, pad_token_id=tokenizer.eos_token_id)

    pairs.append({
        'prompt':   prompt,
        'chosen':   tokenizer.decode(cho_out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True),
        'rejected': tokenizer.decode(rej_out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True),
    })

dpo_dataset = Dataset.from_list(pairs).train_test_split(test_size=0.1, seed=42)
print('DPO dataset created:', len(pairs), 'pairs')

del base_mdl, sft_mdl, sft_base; torch.cuda.empty_cache()

In [ ]:
# CELL 9 — DPO Training
from peft import LoraConfig
from trl import DPOTrainer, DPOConfig

sft_for_dpo = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
sft_for_dpo = PeftModel.from_pretrained(sft_for_dpo, './sft_output/final').merge_and_unload()

dpo_lora = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
)

dpo_config = DPOConfig(
    output_dir='./dpo_output',
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    beta=0.1,
    fp16=True,
    logging_steps=20,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    report_to='mlflow',
    run_name='dpo-qwen2.5',
    max_length=512,
    max_prompt_length=256,
)

dpo_trainer = DPOTrainer(
    model=sft_for_dpo,
    ref_model=None,
    args=dpo_config,
    train_dataset=dpo_dataset['train'],
    eval_dataset=dpo_dataset['test'],
    tokenizer=tokenizer,
    peft_config=dpo_lora,
)

with mlflow.start_run(run_name='dpo'):
    mlflow.log_params({'beta': 0.1, 'lr': 5e-5, 'dpo_pairs': len(pairs)})
    dpo_trainer.train()

dpo_trainer.save_model('./dpo_output/final')
tokenizer.save_pretrained('./dpo_output/final')
print('DPO training complete.')

In [ ]:
# CELL 10 — Evaluate SFT+DPO vs SFT vs Base
dpo_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
dpo_mdl  = PeftModel.from_pretrained(dpo_base, './dpo_output/final')
_, dpo_scores = evaluate(dpo_mdl, tokenizer, test_prompts, references, 'SFT+DPO')
del dpo_mdl, dpo_base; torch.cuda.empty_cache()

print('\n====== FINAL RESULTS — COPY INTO CV ======')
print(f'Base    ROUGE-L: {base_scores["rouge_l"]} | BERTScore: {base_scores["bertscore"]}')
print(f'SFT     ROUGE-L: {sft_scores["rouge_l"]}  | BERTScore: {sft_scores["bertscore"]}')
print(f'SFT+DPO ROUGE-L: {dpo_scores["rouge_l"]}  | BERTScore: {dpo_scores["bertscore"]}')

In [ ]:
# CELL 11 — INT4 Quantisation + latency benchmark
from transformers import BitsAndBytesConfig
import time

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# Save merged DPO model first
dpo_for_quant = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
dpo_for_quant = PeftModel.from_pretrained(dpo_for_quant, './dpo_output/final').merge_and_unload()
dpo_for_quant.save_pretrained('./quantised/merged')
tokenizer.save_pretrained('./quantised/merged')
del dpo_for_quant; torch.cuda.empty_cache()

# Load quantised
quant_mdl = AutoModelForCausalLM.from_pretrained(
    './quantised/merged',
    quantization_config=bnb_config,
    device_map='auto',
)

BENCH_PROMPT = (
    '<|im_start|>system\nYou are a helpful e-commerce customer service assistant.\n<|im_end|>\n'
    '<|im_start|>user\nI want to cancel my order and get a refund.\n<|im_end|>\n'
    '<|im_start|>assistant\n'
)

def bench(mdl, tok, runs=30):
    inp = tok(BENCH_PROMPT, return_tensors='pt').to(mdl.device)
    lats = []
    mdl.eval()
    with torch.no_grad():
        for _ in range(runs):
            t = time.perf_counter()
            mdl.generate(**inp, max_new_tokens=100, pad_token_id=tok.eos_token_id)
            lats.append(time.perf_counter() - t)
    return round(sum(lats)/len(lats)*1000, 1)

def model_size_gb(mdl):
    return round(sum(p.numel()*p.element_size() for p in mdl.parameters())/1e9, 3)

# Benchmark quantised
quant_lat  = bench(quant_mdl, tokenizer)
quant_size = model_size_gb(quant_mdl)

# Load FP16 for comparison
fp16_mdl  = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
fp16_lat  = bench(fp16_mdl, tokenizer)
fp16_size = model_size_gb(fp16_mdl)
del fp16_mdl; torch.cuda.empty_cache()

size_reduction = round((1 - quant_size/fp16_size)*100, 1)
lat_improvement = round((1 - quant_lat/fp16_lat)*100, 1)

print('\n====== QUANTISATION RESULTS — COPY INTO CV ======')
print(f'FP16  size: {fp16_size} GB  | latency: {fp16_lat} ms')
print(f'INT4  size: {quant_size} GB | latency: {quant_lat} ms')
print(f'Size reduction:      {size_reduction}%')
print(f'Latency improvement: {lat_improvement}%')
print('\nCV bullet [A]:', fp16_size, 'GB')
print('CV bullet [B]:', quant_size, 'GB')
print('CV bullet [C]:', lat_improvement, '%')